In [2]:
import gzip
from parselog import *

def itersim(path):
    with gzip.open(path, "rb") as fp:
        for line in fp:
            yield line.decode("utf-8").strip()

its = [
    #itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-none-none/patricia/sim_stdout.gz"),
    #itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-l1fixx2-cap_ptr_new-l1/patricia/sim_stdout.gz"),
    #itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-l1fix-cap_ptr_new-l1/patricia/sim_stdout.gz"),
    #itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-l1fix2-none-none/dijkstra_capchain/sim_stdout.gz"),
    #itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-l1fix2-cap_ptr-l1/dijkstra_capchain/sim_stdout.gz"),
    itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/toooba-cache-test-stride-l1/dijkstra_capchain/sim_stdout.gz"),
    itersim("/local/scratch/ldh35/DE10Pro-cheri-bgas/bluespec/sim-utils/results/capptr-new-cap_ptr-l1/dijkstra_capchain/sim_stdout.gz"),
]
lls = [None]*len(its)

pcExcempt = [
    #0x00000000c0001bfa, # patricia reads mcycle
    #0x00000000c0001e94, # patricia writes mcycle
    #0x00000000c0002478, # patricia writes mcycle
    #0x00000000c0002b68, # patricia writes mcycle
    0x00000000c0001702, # dijkstra_capchain reads mcycle
]

retry = 0
try:
    while True:
        for i, it in enumerate(its):
            while True:
                line = next(it)
                if CRqCreationLine.deduceLineType(line) == CRqCreationLine:
                    if CRqCreationLine(line).isRetry:
                        retry += 1
                        print(f"\rRetries: {retry}", end="")
                if RVFILine.deduceLineType(line) == RVFILine:
                    lls[i] = RVFILine(line)
                    break
        for ll1, ll2 in zip(lls, lls[1:]):
            try:
                assert ll1.rvfi  == ll2.rvfi
                assert ll1.pc    == ll2.pc
                assert ll1.instr == ll2.instr
                assert ll1.pcwd  == ll2.pcwd
                assert ll1.trap  == ll2.trap
                if ll1.pc in pcExcempt:
                    continue
                assert ll1.rd    == ll2.rd
                assert ll1.rwd   == ll2.rwd
                assert ll1.ma    == ll2.ma
                assert ll1.mwd   == ll2.mwd
                assert ll1.mrm   == ll2.mrm
                assert ll1.mwm   == ll2.mwm
            except AssertionError as e:
                print(ll1.line)
                print(ll2.line)
                raise e
except StopIteration:
    pass